# **ACDC In-Domain Training — Faster R-CNN - seed 456**

The notebook includes:
*   Configuration of the Faster R-CNN R50-FPN architecture with COCO pretrained weights
*   Model training and validation on the ACDC dataset
*   Global evaluation on the ACDC test set and weather-specific evaluation on fog, rain, and snow subsets


### **Mount drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **Install Detectron**

In [2]:
# Fix numpy compatibility
!pip install -q --force-reinstall numpy==1.26.4

# Build dependencies
!pip install -q setuptools==68.0.0 wheel cython

#Import detectron from the source
%cd /content

import os

if not os.path.exists("/content/detectron2"):
    !git clone https://github.com/facebookresearch/detectron2.git

%cd /content/detectron2
!python -m pip install --no-build-isolation -e .
%cd /content

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have n

In [ ]:
import os
os.kill(os.getpid(), 9)

In [3]:
import numpy as np
print(np.__version__)

1.26.4


### **Import librairies**

In [4]:
from pathlib import Path
import os
import json
import time
import cv2
import torch
import pandas as pd
import numpy as np

from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_pascal_voc
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.engine import hooks

### **Path configuration & check GPU**

In [5]:
PROJECT_ROOT = Path("/content/drive/MyDrive/Dissertation")

ACDC_VOC_ROOT = PROJECT_ROOT / "Datasets/processed/acdc_voc"
RUNS_ROOT = PROJECT_ROOT / "Runs/faster_rcnn"
OUTPUT_DIR = RUNS_ROOT / "acdc_in_domain_faster_rcnn_seed456"

CLASS_NAMES = [
    "person",
    "bicycle",
    "car",
    "motorcycle",
    "bus",
    "truck"
]

RANDOM_SEED = 456

print("ACDC VOC exists:", ACDC_VOC_ROOT.exists())
print("Train split exists:", (ACDC_VOC_ROOT / "ImageSets/Main/train.txt").exists())
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

ACDC VOC exists: True
Train split exists: True
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


### **Register ACDC VOC datasets**

In [6]:
CLASS_NAMES = ["person", "bicycle", "car", "motorcycle", "bus", "truck"]

datasets = {
    "acdc_train": "train",
    "acdc_val": "val",
    "acdc_test": "test",
    "acdc_test_fog": "test_fog",
    "acdc_test_rain": "test_rain",
    "acdc_test_snow": "test_snow",
}

def reset_detectron2_dataset(name):
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)

    if name in MetadataCatalog.list():
        MetadataCatalog.remove(name)

for dataset_name in datasets:
    reset_detectron2_dataset(dataset_name)

for dataset_name, split_name in datasets.items():
    register_pascal_voc(
        name=dataset_name,
        dirname=str(ACDC_VOC_ROOT),
        split=split_name,
        year="",
        class_names=CLASS_NAMES
    )

print("ACDC datasets registered with custom classes.")

ACDC datasets registered with custom classes.


In [7]:
for dataset_name in datasets:
    print(dataset_name, MetadataCatalog.get(dataset_name).thing_classes)

acdc_train ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
acdc_val ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
acdc_test ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
acdc_test_fog ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
acdc_test_rain ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
acdc_test_snow ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']


### **Mini validation**

In [8]:
for dataset_name in datasets.keys():
    dataset_dicts = DatasetCatalog.get(dataset_name)
    metadata = MetadataCatalog.get(dataset_name)

    print(f"{dataset_name}: {len(dataset_dicts)} images")
    print("classes:", metadata.thing_classes)

acdc_train: 1620 images
classes: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
acdc_val: 540 images
classes: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
acdc_test: 540 images
classes: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
acdc_test_fog: 100 images
classes: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
acdc_test_rain: 340 images
classes: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
acdc_test_snow: 100 images
classes: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']


### **Checkpoints configuration**

In [9]:
IMS_PER_BATCH = 4
EPOCHS = 100

NUM_TRAIN_IMAGES = len(DatasetCatalog.get("acdc_train"))
STEPS_PER_EPOCH = int(np.ceil(NUM_TRAIN_IMAGES / IMS_PER_BATCH))
MAX_ITER = STEPS_PER_EPOCH * EPOCHS

CHECKPOINT_PERIOD = 5000
EVAL_PERIOD = STEPS_PER_EPOCH * 5   # validation toutes les 5 epochs

print("Train images:", NUM_TRAIN_IMAGES)
print("Steps per epoch:", STEPS_PER_EPOCH)
print("MAX_ITER:", MAX_ITER)
print("Checkpoint period:", CHECKPOINT_PERIOD)
print("Eval period:", EVAL_PERIOD)

Train images: 1620
Steps per epoch: 405
MAX_ITER: 40500
Checkpoint period: 5000
Eval period: 2025


### **trainer with best checkpoint**

In [10]:
class TrainerWithBestCheckpoint(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "eval", dataset_name)
        return COCOEvaluator(dataset_name, cfg, False, output_folder)

    def build_hooks(self):
        hook_list = super().build_hooks()

        hook_list.insert(
            -1,
            hooks.BestCheckpointer(
                self.cfg.TEST.EVAL_PERIOD,
                DetectionCheckpointer(self.model, self.cfg.OUTPUT_DIR),
                "bbox/AP",
                mode="max",
                file_prefix="model_best"
            )
        )

        return hook_list

### **Config**

In [11]:
cfg = get_cfg()

cfg.merge_from_file(
    model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml")
)

cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"
)

cfg.DATASETS.TRAIN = ("acdc_train",)
cfg.DATASETS.TEST = ("acdc_val",)

cfg.DATALOADER.NUM_WORKERS = 2

cfg.SOLVER.IMS_PER_BATCH = IMS_PER_BATCH
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = MAX_ITER
cfg.SOLVER.STEPS = []
cfg.SOLVER.CHECKPOINT_PERIOD = CHECKPOINT_PERIOD

cfg.TEST.EVAL_PERIOD = EVAL_PERIOD

cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(CLASS_NAMES)
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5

cfg.SEED = RANDOM_SEED
cfg.OUTPUT_DIR = str(OUTPUT_DIR)

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

with open(OUTPUT_DIR / "config.yaml", "w") as f:
    f.write(cfg.dump())

print("Config ready.")
print("Output:", cfg.OUTPUT_DIR)

Config ready.
Output: /content/drive/MyDrive/Dissertation/Runs/faster_rcnn/acdc_in_domain_faster_rcnn_seed456


### **Train**

In [12]:
trainer = TrainerWithBestCheckpoint(cfg)
trainer.resume_or_load(resume=True)
trainer.train()

[07/16 17:34:11 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

model_final_280758.pkl: 167MB [00:01, 138MB/s]                           
roi_heads.box_predictor.bbox_pred.{bias, weight}
roi_heads.box_predictor.cls_score.{bias, weight}


[07/16 17:34:15 d2.engine.train_loop]: Starting training from iteration 0


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0716 17:34:19.157000 7000 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


[07/16 17:34:37 d2.utils.events]:  eta: 9:24:18  iter: 19  total_loss: 2.758  loss_cls: 2.175  loss_box_reg: 0.5021  loss_rpn_cls: 0.03126  loss_rpn_loc: 0.09134    time: 0.9931  last_time: 0.6863  data_time: 0.7832  last_data_time: 0.5220   lr: 4.9953e-06  max_mem: 4082M
[07/16 17:34:57 d2.utils.events]:  eta: 9:05:06  iter: 39  total_loss: 2.645  loss_cls: 2.038  loss_box_reg: 0.4302  loss_rpn_cls: 0.03556  loss_rpn_loc: 0.07867    time: 0.8900  last_time: 0.8213  data_time: 0.6360  last_data_time: 0.6611   lr: 9.9902e-06  max_mem: 4082M
[07/16 17:35:14 d2.utils.events]:  eta: 9:15:48  iter: 59  total_loss: 2.459  loss_cls: 1.782  loss_box_reg: 0.46  loss_rpn_cls: 0.04378  loss_rpn_loc: 0.09842    time: 0.8690  last_time: 0.6703  data_time: 0.6687  last_data_time: 0.5219   lr: 1.4985e-05  max_mem: 4082M
[07/16 17:35:30 d2.utils.events]:  eta: 9:08:55  iter: 79  total_loss: 2.008  loss_cls: 1.425  loss_box_reg: 0.4138  loss_rpn_cls: 0.04395  loss_rpn_loc: 0.1289    time: 0.8590  last_

In [13]:
for file in sorted(OUTPUT_DIR.glob("*.pth")):
    print(file.name, f"{file.stat().st_size / 1e6:.1f} MB")

if (OUTPUT_DIR / "last_checkpoint").exists():
    print("\nlast_checkpoint content:")
    print((OUTPUT_DIR / "last_checkpoint").read_text())

model_0004999.pth 330.2 MB
model_0009999.pth 330.2 MB
model_0014999.pth 330.2 MB
model_0019999.pth 330.2 MB
model_0024999.pth 330.2 MB
model_0029999.pth 330.2 MB
model_0034999.pth 330.2 MB
model_0039999.pth 330.2 MB
model_best.pth 165.8 MB
model_final.pth 330.2 MB

last_checkpoint content:
model_best.pth


### **Evaluation with model_best**

In [14]:
BEST_MODEL = OUTPUT_DIR / "model_best.pth"

if BEST_MODEL.exists():
    cfg.MODEL.WEIGHTS = str(BEST_MODEL)
    print("Using best model:", BEST_MODEL)
else:
    cfg.MODEL.WEIGHTS = str(OUTPUT_DIR / "model_final.pth")
    print("Using final model instead.")

cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
predictor = DefaultPredictor(cfg)

Using best model: /content/drive/MyDrive/Dissertation/Runs/faster_rcnn/acdc_in_domain_faster_rcnn_seed456/model_best.pth
[07/16 20:04:05 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /content/drive/MyDrive/Dissertation/Runs/faster_rcnn/acdc_in_domain_faster_rcnn_seed456/model_best.pth ...


In [15]:
eval_datasets = {
    "global": "acdc_test",
    "fog": "acdc_test_fog",
    "rain": "acdc_test_rain",
    "snow": "acdc_test_snow",
}

results_rows = []

for condition, dataset_name in eval_datasets.items():
    print(f"\nEvaluating {condition.upper()}...")

    evaluator = COCOEvaluator(
        dataset_name,
        cfg,
        False,
        output_dir=str(OUTPUT_DIR / "evaluation" / condition)
    )

    loader = build_detection_test_loader(cfg, dataset_name)
    eval_results = inference_on_dataset(predictor.model, loader, evaluator)

    bbox = eval_results["bbox"]

    results_rows.append({
        "dataset": "ACDC",
        "model": "Faster R-CNN",
        "experiment": "in_domain",
        "seed": RANDOM_SEED,
        "condition": condition,
        "mAP50-95": bbox["AP"] / 100,
        "mAP50": bbox["AP50"] / 100,
        "mAP75": bbox["AP75"] / 100,
    })

results_df = pd.DataFrame(results_rows)
results_df


Evaluating GLOBAL...
WARNING [07/16 20:04:07 d2.evaluation.coco_evaluation]: COCO Evaluator instantiated using config, this is deprecated behavior. Please pass in explicit arguments instead.
[07/16 20:04:07 d2.evaluation.coco_evaluation]: Trying to convert 'acdc_test' to COCO format ...
[07/16 20:04:07 d2.data.datasets.coco]: Converting annotations of dataset 'acdc_test' to COCO format ...)
[07/16 20:04:08 d2.data.datasets.coco]: Converting dataset dicts into COCO format
[07/16 20:04:09 d2.data.datasets.coco]: Conversion finished, #images: 540, #annotations: 3358
[07/16 20:04:09 d2.data.datasets.coco]: Caching COCO format annotations at '/content/drive/MyDrive/Dissertation/Runs/faster_rcnn/acdc_in_domain_faster_rcnn_seed456/evaluation/global/acdc_test_coco_format.json' ...
[07/16 20:04:10 d2.data.build]: Distribution of instances among all 6 categories:
|  category  | #instances   |  category  | #instances   |  category  | #instances   |
|:----------:|:-------------|:----------:|:----

,dataset,model,experiment,seed,condition,mAP50-95,mAP50,mAP75
0,ACDC,Faster R-CNN,in_domain,456,global,0.416332,0.597323,0.448921
1,ACDC,Faster R-CNN,in_domain,456,fog,0.552021,0.726634,0.584098
2,ACDC,Faster R-CNN,in_domain,456,rain,0.362602,0.531106,0.388452
3,ACDC,Faster R-CNN,in_domain,456,snow,0.497758,0.693990,0.557561


### **FPS**

In [ ]:
def get_image_paths_from_split(voc_root, split_name):
    split_file = voc_root / "ImageSets" / "Main" / f"{split_name}.txt"

    with open(split_file, "r") as f:
        image_ids = [line.strip() for line in f if line.strip()]

    return [
        voc_root / "JPEGImages" / f"{image_id}.jpg"
        for image_id in image_ids
        if (voc_root / "JPEGImages" / f"{image_id}.jpg").exists()
    ]


def measure_fps(predictor, image_paths, warmup=20):
    images = []

    for p in image_paths:
        img = cv2.imread(str(p))
        if img is not None:
            images.append(img)

    for img in images[:warmup]:
        _ = predictor(img)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    times = []

    for img in images:
        start = time.perf_counter()
        _ = predictor(img)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end = time.perf_counter()
        times.append(end - start)

    avg_ms = np.mean(times) * 1000
    fps = 1000 / avg_ms

    return avg_ms, fps

In [ ]:
split_mapping = {
    "global": "test",
    "fog": "test_fog",
    "rain": "test_rain",
    "snow": "test_snow",
}

for idx, row in results_df.iterrows():
    condition = row["condition"]
    split_name = split_mapping[condition]

    image_paths = get_image_paths_from_split(ACDC_VOC_ROOT, split_name)

    inference_ms, fps = measure_fps(
        predictor,
        image_paths,
        warmup=20
    )

    results_df.loc[idx, "Inference_ms_per_image"] = inference_ms
    results_df.loc[idx, "FPS"] = fps

results_df

,dataset,model,experiment,seed,condition,mAP50-95,mAP50,mAP75,Inference_ms_per_image,FPS
0,ACDC,Faster R-CNN,in_domain,123,global,0.428254,0.612108,0.455618,61.770184,16.189040
1,ACDC,Faster R-CNN,in_domain,123,fog,0.572156,0.764261,0.621466,61.786938,16.184651
2,ACDC,Faster R-CNN,in_domain,123,rain,0.374499,0.544613,0.388124,60.997793,16.394036
3,ACDC,Faster R-CNN,in_domain,123,snow,0.515350,0.697251,0.567666,61.904689,16.153865


### **Save results**

In [ ]:
results_csv = OUTPUT_DIR / "acdc_faster_rcnn_in_domain_results_summary.csv"
results_json = OUTPUT_DIR / "acdc_faster_rcnn_in_domain_results_summary.json"

results_df.to_csv(results_csv, index=False)

with open(results_json, "w") as f:
    json.dump(results_df.to_dict(orient="records"), f, indent=4)

print("Saved:")
print(results_csv)
print(results_json)

Saved:
/content/drive/MyDrive/Dissertation/Runs/faster_rcnn/acdc_in_domain_faster_rcnn_seed123/acdc_faster_rcnn_in_domain_results_summary.csv
/content/drive/MyDrive/Dissertation/Runs/faster_rcnn/acdc_in_domain_faster_rcnn_seed123/acdc_faster_rcnn_in_domain_results_summary.json
